# benchmark-CPTAC-MCAR
6.16.25

Can we get the MSEs for the Lupine and DreamAI reconstructions on CPTAC MCAR split? 

In [1]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Import my utils module. Need my `mse_func`
sys.path.append("../../../bin/")
from utils import *

# plotting templates
sns.set(context="talk", style="ticks") 
sns.set_palette("tab10")

#### Configs

In [2]:
data_path="../data/"
results_path="../results/"
lupine_ensembled_path="lupine-ensemble-CPTAC-MCAR.csv"

cohorts=["BRCA", "CCRCC", "COAD", "GBM", "HGSC", "UCEC",
         "HNSCC", "LSCC", "LUAD", "PDAC"]

#### Functions 

In [3]:
def mse_func(x_mat, y_mat):
    """
    Calculate the MSE for two matricies with missing values. Each
    matrix can contain MVs, in the form of np.nans
    
    Parameters
    ----------
    x_mat : np.ndarray, 
        The first matrix 
    y_mat : np.ndarray, T
        he second matrix
    
    Returns
    -------
    float, the mean squared error between values present 
            across both matrices
    """
    x_rav = x_mat.ravel()
    y_rav = y_mat.ravel()
    missing = np.isnan(x_rav) | np.isnan(y_rav)
    mse = np.sum((x_rav[~missing] - y_rav[~missing])**2)
    
    return mse / np.sum(~missing)

#### Read in the Lupine recon ensembled matrix
And get the associated row and column IDs. 

In [4]:
lupine_recon = pd.read_csv(results_path+lupine_ensembled_path, index_col=0)
lupine_train_pd = pd.read_csv(data_path+"lupine-train-pandas.csv", index_col=0)

print(lupine_recon.shape)
print(lupine_train_pd.shape)

lupine_recon.index = lupine_train_pd.index
lupine_recon.columns = lupine_train_pd.columns

(18984, 1905)
(18984, 1905)


#### Get the test set accuracy for Lupine and DreamAI
For each individual cohort. 

In [5]:
for cohort in cohorts: 
    print(cohort)
    # Read in the DreamAI imputed matrix, for the current cohort
    cohort_dream_imputed = pd.read_csv(results_path + cohort + "_dreamAI_recon.csv", index_col=0)
    cohort_anno_df = pd.read_csv(data_path + cohort + "_dreamAI_annotated.csv", index_col=0)

    cohort_dream_imputed.index = cohort_anno_df.index
    cohort_dream_imputed.columns = cohort_anno_df.columns

    # Pair down the Lupine imputed matrix to the same rows & columns
    cohort_lupine_imputed = lupine_recon[cohort_dream_imputed.columns]
    cohort_lupine_imputed = cohort_lupine_imputed.loc[cohort_dream_imputed.index]

    # Read in the corresponding test set dataframe
    cohort_test_df = pd.read_csv(data_path + cohort + "_test_dreamAI.csv", index_col=0)

    curr_lupine_err = mse_func(np.array(cohort_lupine_imputed), np.array(cohort_test_df))
    curr_dream_err = mse_func(np.array(cohort_dream_imputed), np.array(cohort_test_df))

    print(f"Lupine test error: {np.around(curr_lupine_err, 3)}")
    print(f"DreamAI test error: {np.around(curr_dream_err, 3)}")
    print(" ")

BRCA
Lupine test error: 0.028
DreamAI test error: 0.026
 
CCRCC
Lupine test error: 0.032
DreamAI test error: 0.036
 
COAD
Lupine test error: 0.025
DreamAI test error: 0.024
 
GBM
Lupine test error: 0.028
DreamAI test error: 0.027
 
HGSC
Lupine test error: 0.042
DreamAI test error: 0.042
 
UCEC
Lupine test error: 0.033
DreamAI test error: 0.032
 
HNSCC
Lupine test error: 0.034
DreamAI test error: 0.035
 
LSCC
Lupine test error: 0.024
DreamAI test error: 0.023
 
LUAD
Lupine test error: 0.026
DreamAI test error: 0.026
 
PDAC
Lupine test error: 0.023
DreamAI test error: 0.02
 


#### Repeat, for KNN impute

In [6]:
for cohort in cohorts: 
    print(cohort)
    # Read in the DreamAI imputed matrix, for the current cohort
    cohort_knn_imputed = pd.read_csv(results_path + cohort + "_train_knn_imputed.csv", index_col=0)
    cohort_anno_df = pd.read_csv(data_path + cohort + "_dreamAI_annotated.csv", index_col=0)

    cohort_knn_imputed.index = cohort_anno_df.index
    cohort_knn_imputed.columns = cohort_anno_df.columns

    # Read in the corresponding test set dataframe
    cohort_test_df = pd.read_csv(data_path + cohort + "_test_dreamAI.csv", index_col=0)

    #curr_lupine_err = mse_func(np.array(cohort_lupine_imputed), np.array(cohort_test_df))
    curr_knn_err = mse_func(np.array(cohort_knn_imputed), np.array(cohort_test_df))

    print(f"kNN test error: {np.around(curr_knn_err, 3)}")
    print(" ")

BRCA
kNN test error: 0.035
 
CCRCC
kNN test error: 0.065
 
COAD
kNN test error: 0.031
 
GBM
kNN test error: 0.032
 
HGSC
kNN test error: 0.051
 
UCEC
kNN test error: 0.05
 
HNSCC
kNN test error: 0.062
 
LSCC
kNN test error: 0.029
 
LUAD
kNN test error: 0.04
 
PDAC
kNN test error: 0.03
 


#### Repeat for MissForest

In [7]:
for cohort in cohorts: 
    print(cohort)
    # Read in the DreamAI imputed matrix, for the current cohort
    cohort_mf_imputed = pd.read_csv(results_path + cohort + "_train_mf_imputed.csv", index_col=0)
    cohort_mf_imputed = cohort_mf_imputed.T
    cohort_anno_df = pd.read_csv(data_path + cohort + "_dreamAI_annotated.csv", index_col=0)

    cohort_mf_imputed.index = cohort_anno_df.index
    cohort_mf_imputed.columns = cohort_anno_df.columns

    # Read in the corresponding test set dataframe
    cohort_test_df = pd.read_csv(data_path + cohort + "_test_dreamAI.csv", index_col=0)

    #curr_lupine_err = mse_func(np.array(cohort_lupine_imputed), np.array(cohort_test_df))
    curr_mf_err = mse_func(np.array(cohort_mf_imputed), np.array(cohort_test_df))

    print(f"MissForest test error: {np.around(curr_mf_err, 3)}")
    print(" ")

BRCA
MissForest test error: 0.029
 
CCRCC
MissForest test error: 0.038
 
COAD
MissForest test error: 0.028
 
GBM
MissForest test error: 0.028
 
HGSC
MissForest test error: 0.043
 
UCEC
MissForest test error: 0.033
 
HNSCC
MissForest test error: 0.039
 
LSCC
MissForest test error: 0.025
 
LUAD
MissForest test error: 0.028
 
PDAC
MissForest test error: 0.025
 


#### Init a dataframe to hold the results

In [ ]:
bench_res = pd.DataFrame(columns=["cohort", "dream_error", "lupine_error"])
bench_res["cohort"] = ["BRCA", "CCRCC", "COAD", "GBM", "HGSC", 
                 "HNSCC", "LSCC", "LUAD", "PDAC", "UCEC"]
bench_res["lupine_error"] = [0.028, 0.032, 0.025, 0.028, 0.042, 0.033, 0.034, 0.024, 0.026, 0.023]
bench_res["dream_error"] = [0.026, 0.036, 0.024, 0.027, 0.042, 0.032, 0.035, 0.023, 0.026, 0.020]

#### Add in basline impute methods

In [ ]:
baseline_res = []

for cohort in cohorts:
    train_mat = pd.read_csv(data_path + cohort + "_train_dreamAI.csv", index_col=0)
    test_mat = pd.read_csv(data_path + cohort + "_test_dreamAI.csv", index_col=0)

    train = np.array(train_mat)
    test = np.array(test_mat)

    #mean_res = mean_impute(train, test)
    #min_res = min_impute(train, test)

    # Impute with Gaussian random sample
    # axis=0 will apply the impute function to every column 
    rd_imputed = np.apply_along_axis(random_draw_impute, 0, train)

    # Compute the test error for Gaussian random sample
    rd_error = mse_func(rd_imputed, test)

    curr_res = {
        "cohort": cohort, 
        "rand_draw_err": rd_error, 
    }
    baseline_res.append(curr_res)
    
baseline_res = pd.DataFrame.from_dict(baseline_res)

#### Add in baseline errors to the existing results dataframe

In [ ]:
# bench_res["mean_error"] = baseline_res["mean_error"]
# bench_res["min_error"] = baseline_res["min_res"]
bench_res["rand_draw_error"] = baseline_res["rand_draw_err"]

#### Add in kNN and MissForest results

In [ ]:
bench_res["kNN_error"] = [0.035, 0.065, 0.031, 0.032, 0.051, 0.050, 0.062, 0.029, 0.04, 0.03]
bench_res["missForest_error"] = [0.029, 0.038, 0.028, 0.028, 0.043, 0.033, 0.039, 0.025, 0.028, 0.025]

#### Scatterplot, just two panels 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(8,4))

# LUPINE VS LOW VALUE IMPUTE
sns.scatterplot(
    data=bench_res, 
    x="rand_draw_error", 
    y="lupine_error", 
    alpha=1.0, 
    zorder=2, 
    s=75,
    ax=ax1,
    color="#d62728",
)
#ax1.set_title("")
ax1.set_xlabel("Random Sampling", labelpad=16, fontsize=18)
ax1.set_ylabel("Lupine", labelpad=12, fontsize=18)

# Add diagonal line
ax1_min = -0.25
ax1_max = 4.75
x = np.linspace(ax1_min, ax1_max, 100)
y = x
ax1.plot(x, y, color="black", alpha=1.0, zorder=1, linewidth=2)

ax1.set_xlim(-0.5, 5)
ax1.set_ylim(-0.5, 5)

# LUPINE VS DREAM-AI
sns.scatterplot(
    data=bench_res,
    x="dream_error", 
    y="lupine_error", 
    alpha=1.0, 
    zorder=2, 
    s=75,
    ax=ax2,
    label="DreamAI",
)
# LUPINE VS KNN IMPUTE
sns.scatterplot(
    data=bench_res,
    x="kNN_error", 
    y="lupine_error", 
    alpha=1.0, 
    zorder=2, 
    s=75,
    ax=ax2,
    label="kNN",
)
# LUPINE VS MISSFOREST IMPUTE
sns.scatterplot(
    data=bench_res,
    x="missForest_error", 
    y="lupine_error", 
    alpha=1.0, 
    zorder=2, 
    s=75,
    ax=ax2,
    label="MissForest",
)
#ax2.set_title("foobar")
ax2.set_xlabel("Other Method", labelpad=16, fontsize=18)
ax2.set_ylabel("", labelpad=16, fontsize=16)

ax2.set_xlim(0.01, 0.07)
ax2.set_ylim(0.01, 0.07)

# Add diagonal line
ax2_min = 0.013
ax2_max = 0.068
x = np.linspace(ax2_min, ax2_max, 100)
y = x
ax2.plot(x, y, color="black", alpha=1.0, zorder=1, linewidth=2)

ax2.legend(title="", edgecolor="k", prop={'size': 10}, loc="upper left")

# GENERAL SUBPLOT CONFIGS

ax1.tick_params(axis='both', which='major', labelsize=12)
ax2.tick_params(axis='both', which='major', labelsize=12)

ax1.locator_params(axis='x', min_n_ticks=2, nbins=4, tight="true")
ax1.locator_params(axis='y', min_n_ticks=2, nbins=4, tight="true")

ax2.locator_params(axis='x', min_n_ticks=3, nbins=4, tight="true")
ax2.locator_params(axis='y', min_n_ticks=3, nbins=4, tight="true")

fig.tight_layout()
#plt.show()
plt.savefig("../figures/CPTAC-MCAR-benchmarking.pdf", bbox_inches="tight")
#plt.savefig("../figures/CPTAC-MCAR-benchmarking.png", bbox_inches="tight", dpi=250)